In [11]:
import os
from datetime import datetime, timedelta, timezone
import pandas as pd
from pathlib import Path
import pyarrow
import numpy as np
import matplotlib.pyplot as plt
from zoneinfo import ZoneInfo
import json, requests
from __future__ import annotations
import time


## Get Token Id for Each Game

In [8]:
def get_clob_token_ids(slug: str) -> dict:
    """Returns {'yes': '<token_id>', 'no': '<token_id>'} for a Polymarket slug."""
    r = requests.get(
        f"https://gamma-api.polymarket.com/markets/slug/{slug}",
        params={"include_tag": "true"}, timeout=20,
    )
    r.raise_for_status()
    m = r.json()
    ids = m["clobTokenIds"]
    if isinstance(ids, str):           # gamma returns it stringified
        ids = json.loads(ids)
    outcomes = m.get("outcomes")
    if isinstance(outcomes, str):
        outcomes = json.loads(outcomes)
    # Sanity-check the YES/NO ordering using the outcomes field.
    print(f"question:  {m['question']}")
    print(f"outcomes:  {outcomes}      (index 0=YES, 1=NO)")
    print(f"conditionId: {m['conditionId']}")
    return {"yes": ids[0], "no": ids[1]}

tokens = get_clob_token_ids("nba-lal-hou-2026-04-24")
tokens

question:  Lakers vs. Rockets
outcomes:  ['Lakers', 'Rockets']      (index 0=YES, 1=NO)
conditionId: 0x99e70457e21d76b219505ba2743bc87362a5bba2b478b536720e599418d5a7ae


{'yes': '80723115786902560093562209850109820656482936758561314212132210830217888670020',
 'no': '107793600296183640087234882966799319361195949492981078991430273990508566137204'}

## Get Order Book Data

In [ ]:
API_URL = "https://api.predexon.com/v2/polymarket/orderbooks"

def fetch_orderbooks_streaming(token_id: str, start_ms: int, end_ms: int,
                               api_key: str, out_path: Path, *,
                               limit: int = 200,
                               sleep_between_pages: float = 0.05,
                               max_retries: int = 3) -> int:
    """Page through [start_ms, end_ms] and APPEND each snapshot to `out_path`
    as JSONL. Returns total snapshots written. Each page is flushed+fsync'd,
    so a crash loses at most the snapshots in the in-flight page."""
    headers = {"x-api-key": api_key}
    params = {
        "token_id":   token_id,
        "start_time": start_ms,
        "end_time":   end_ms,
        "limit":      limit,
    }
    n_total = 0
    page = 0
    with out_path.open("w") as f:
        while True:
            for attempt in range(max_retries):
                try:
                    r = requests.get(API_URL, headers=headers, params=params, timeout=30)
                    r.raise_for_status()
                    break
                except requests.RequestException as e:
                    if attempt == max_retries - 1:
                        raise
                    wait = 2 ** attempt
                    print(f"  [retry] {e}; sleeping {wait}s")
                    time.sleep(wait)
            data = r.json()
            snaps = data.get("snapshots", [])
            pagination = data.get("pagination", {})

            # ---- write THIS page to disk before doing anything else ----
            for s in snaps:
                f.write(json.dumps(s) + "\n")
            f.flush()
            os.fsync(f.fileno())  # belt + suspenders: survives kernel-level crash

            n_total += len(snaps)
            page += 1
            print(f"  page {page:>3}: +{len(snaps):>4} (total {n_total:,}, "
                  f"has_more={pagination.get('has_more')})")
            if not pagination.get("has_more"):
                break
            next_key = pagination.get("pagination_key")
            if not next_key:
                break
            params["pagination_key"] = next_key
            time.sleep(sleep_between_pages)
    return n_total